In [3]:
import pandas as pd
import numpy as np 
import scanpy as sc 
import os
import scrublet as scr 
import glob

In [4]:
# find . -type f | sort

In [5]:
study_files = {
    "Geistlinger2020": {
        "matrix": "Data_Geistlinger2020_Ovarian/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Geistlinger2020_Ovarian/Genes.txt",
        "cells":  "Data_Geistlinger2020_Ovarian/Cells.csv",
        "samples": "Data_Geistlinger2020_Ovarian/Samples.csv",
        "units": "UMI"
    },
    "Izar2020_10X": {
        "matrix": "Data_Izar2020_Ovarian/10X/Exp_data_TPM.mtx",
        "genes":  "Data_Izar2020_Ovarian/10X/Genes.txt",
        "cells":  "Data_Izar2020_Ovarian/10X/Cells.csv",
        "samples": "Data_Izar2020_Ovarian/Samples.csv",
        "units": "TPM"
    },
    "Izar2020_SmartSeq2": {
        "matrix": "Data_Izar2020_Ovarian/SmartSeq2/Exp_data_TPM.mtx",
        "genes":  "Data_Izar2020_Ovarian/SmartSeq2/Genes.txt",
        "cells":  "Data_Izar2020_Ovarian/SmartSeq2/Cells.csv",
        "samples": "Data_Izar2020_Ovarian/Samples.csv",
        "units": "TPM"
    },
    "Nath2021_10X": {
        "matrix": "Data_Nath2021_Ovarian/10X/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Nath2021_Ovarian/10X/Genes.txt",
        "cells":  "Data_Nath2021_Ovarian/10X/Cells.csv",
        "samples": "Data_Nath2021_Ovarian/Samples.csv",
        "units": "UMI"
    },
    "Nath2021_iCell8": {
        "matrix": "Data_Nath2021_Ovarian/iCell8/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Nath2021_Ovarian/iCell8/Genes.txt",
        "cells":  "Data_Nath2021_Ovarian/iCell8/Cells.csv",
        "samples": "Data_Nath2021_Ovarian/Samples.csv",
        "units": "UMI"
    },
    "Olalekan2021": {
        "matrix": "Data_Olalekan2021_Ovarian/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Olalekan2021_Ovarian/Genes.txt",
        "cells":  "Data_Olalekan2021_Ovarian/Cells.csv",
        "samples": "Data_Olalekan2021_Ovarian/Samples.csv",
        "units": "UMI"
    },
    "Olbrecht2021": {
        "matrix": "Data_Olbrecht2021_Ovarian/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Olbrecht2021_Ovarian/Genes.txt",
        "cells":  "Data_Olbrecht2021_Ovarian/Cells.csv",
        "samples": "Data_Olbrecht2021_Ovarian/Samples.csv",
        "units": "UMI"
    },
    "Qian2020": {
        "matrix": "Data_Qian2020_Ovarian/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Qian2020_Ovarian/Genes.txt",
        "cells":  "Data_Qian2020_Ovarian/Cells.csv",
        "samples": "Data_Qian2020_Ovarian/Samples.csv",
        "units": "UMI"
    },
    "Regner2021": {
        "matrix": "Data_Regner2021_Ovarian/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Regner2021_Ovarian/Genes.txt",
        "cells":  "Data_Regner2021_Ovarian/Cells.csv",
        "samples": "Data_Regner2021_Ovarian/Samples.csv",
        "units": "UMI"
    },
    "Shih2018": {
        "matrix": "Data_Shih2018_Ovarian/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Shih2018_Ovarian/Genes.txt",
        "cells":  "Data_Shih2018_Ovarian/Cells.csv",
        "samples": "Data_Shih2018_Ovarian/Samples.csv",
        "units": "UMI"
    },
    "Tang-Huau2018": {
        "matrix": "Data_Tang-Huau2018_Ovarian/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Tang-Huau2018_Ovarian/Genes.txt",
        "cells":  "Data_Tang-Huau2018_Ovarian/Cells.csv",
        "samples": "Data_Tang-Huau2018_Ovarian/Samples.csv",
        "units": "UMI"
    },
    "Zhang2019": {
        "matrix": "Data_Zhang2019_Ovarian/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Zhang2019_Ovarian/Genes.txt",
        "cells":  "Data_Zhang2019_Ovarian/Cells.csv",
        "samples": "Data_Zhang2019_Ovarian/Samples.csv",
        "units": "UMI"
    },
    "Zhang2022": {
        "matrix": "Data_Zhang2022_Ovarian/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Zhang2022_Ovarian/Genes.txt",
        "cells":  "Data_Zhang2022_Ovarian/Cells.csv",
        "samples": "Data_Zhang2022_Ovarian/Samples.csv",
        "units": "UMI"
    }
}

In [6]:
study_name = 'Geistlinger2020'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        # Handle duplicate indices in Samples.csv
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

# 11. Save
out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 41,031 cells × 15,328 genes
   cell_type: 1704 NaN
   cell_type: 1704 suspicious
Flagged 1704 cells
After metadata drop: 39,327 cells
   Added cancer_type column. Unique values: ['Ovarian Cancer']
After gene filter: 38,983 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.56
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 31.6%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 45.6 seconds
After doublet removal: 38,965 cells
After MT filter: 38,965 cells
Normalised (UMI) – max value 13.55


In [7]:
study_name = 'Izar2020_10X'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        # Handle duplicate indices in Samples.csv
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

# 11. Save
out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 9,609 cells × 11,548 genes
   cell_type: 181 NaN
   cell_type: 181 suspicious
Flagged 181 cells
After metadata drop: 9,428 cells
   Added cancer_type column. Unique values: [nan]
After gene filter: 9,404 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.33
Detected doublet rate = 0.7%
Estimated detectable doublet fraction = 49.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.4%
Elapsed time: 6.8 seconds
After doublet removal: 9,340 cells
After MT filter: 9,306 cells
Normalised (TPM) – max value 11.31


In [8]:
study_name = 'Izar2020_SmartSeq2'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        # Handle duplicate indices in Samples.csv
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

# 11. Save
out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 1,179 cells × 23,686 genes
Flagged 0 cells
   Added cancer_type column. Unique values: [nan]
After gene filter: 127 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...


/home/pandeyps/.pyenv/versions/3.11.9/lib/python3.11/site-packages/scrublet/helper_functions.py:239: RuntimeWarning: invalid value encountered in log
  gLog = lambda input: np.log(input[1] * np.exp(-input[0]) + input[2])
/home/pandeyps/.pyenv/versions/3.11.9/lib/python3.11/site-packages/scrublet/helper_functions.py:252: RuntimeWarning: invalid value encountered in sqrt
  CV_input = np.sqrt(b);


Calculating doublet scores...
Automatically set threshold at doublet score = 0.16
Detected doublet rate = 1.6%
Estimated detectable doublet fraction = 15.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 10.0%
Elapsed time: 0.2 seconds
After doublet removal: 125 cells
After MT filter: 125 cells
Normalised (TPM) – max value 5.52


In [9]:
study_name = 'Nath2021_10X'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 41,581 cells × 21,994 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Ovarian Cancer']
After gene filter: 39,275 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.56
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 14.6%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.2%
Elapsed time: 48.7 seconds
After doublet removal: 39,266 cells
After MT filter: 23,888 cells
Normalised (UMI) – max value 13.49


In [10]:
study_name = 'Nath2021_iCell8'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 148 cells × 19,371 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Ovarian Cancer']
After gene filter: 148 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.16
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 1.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 0.2 seconds
After doublet removal: 148 cells
After MT filter: 148 cells
Normalised (UMI) – max value 10.80


In [11]:
study_name = 'Olalekan2021'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 9,885 cells × 16,041 genes
   cell_type: 66 NaN
   cell_type: 66 suspicious
Flagged 66 cells
After metadata drop: 9,819 cells
   Added cancer_type column. Unique values: ['Ovarian Cancer' 'Carcinosarcoma']
After gene filter: 9,718 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.61
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 3.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.3%
Elapsed time: 7.4 seconds
After doublet removal: 9,717 cells
After MT filter: 9,717 cells
Normalised (UMI) – max value 13.18


In [12]:
study_name = 'Olbrecht2021'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 15,528 cells × 33,694 genes
   cell_type: 551 NaN
   cell_type: 551 suspicious
Flagged 551 cells
After metadata drop: 14,977 cells
   Added cancer_type column. Unique values: ['Ovarian Cancer']
After gene filter: 13,945 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.66
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 5.5%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.1%
Elapsed time: 13.7 seconds
After doublet removal: 13,937 cells
After MT filter: 13,464 cells
Normalised (UMI) – max value 13.51


In [13]:
study_name = 'Qian2020'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 16,951 cells × 22,276 genes
   cell_type: 1423 NaN
   cell_type: 1423 suspicious
Flagged 1423 cells
After metadata drop: 15,528 cells
   Added cancer_type column. Unique values: ['Ovarian Cancer']
After gene filter: 15,528 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.64
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 10.3%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.5%
Elapsed time: 14.2 seconds
After doublet removal: 15,520 cells
After MT filter: 15,520 cells
Normalised (UMI) – max value 13.54


In [14]:
study_name = 'Regner2021'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 65,144 cells × 24,516 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Endometrial Cancer' 'Ovarian Cancer' 'Carcinosarcoma'
 'Gastrointestinal Stromal Tumor']
After gene filter: 63,488 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.74
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 9.6%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 308.2 seconds
After doublet removal: 63,486 cells
After MT filter: 55,449 cells
Normalised (UMI) – max value 13.50


In [15]:
study_name = 'Shih2018'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 3,066 cells × 26,364 genes
   cell_type: 1157 NaN
   cell_type: 1157 suspicious
Flagged 1157 cells
After metadata drop: 1,909 cells
After gene filter: 1,907 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.42
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 2.6%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 4.0%
Elapsed time: 1.6 seconds
After doublet removal: 1,905 cells
After MT filter: 1,905 cells
Normalised (UMI) – max value 12.84


In [16]:
study_name = 'Tang-Huau2018'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 8,404 cells × 20,939 genes
Flagged 0 cells
After gene filter: 8,356 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.59
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 6.2%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.6%
Elapsed time: 7.7 seconds
After doublet removal: 8,353 cells
After MT filter: 8,342 cells
Normalised (UMI) – max value 13.50


In [17]:
study_name = 'Zhang2019'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 4,848 cells × 24,410 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Ovarian Cancer']
After gene filter: 4,299 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.37
Detected doublet rate = 0.3%
Estimated detectable doublet fraction = 39.9%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.8%
Elapsed time: 3.7 seconds
After doublet removal: 4,286 cells
After MT filter: 4,092 cells
Normalised (UMI) – max value 13.41


In [18]:
study_name = 'Zhang2022'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Ovarian'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 51,786 cells × 32,847 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Ovarian Cancer']
After gene filter: 50,870 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.79
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.8%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 105.3 seconds
After doublet removal: 50,870 cells
After MT filter: 50,870 cells
Normalised (UMI) – max value 13.51


In [19]:
INPUT_PATTERN = "*.h5ad"   

files = sorted(glob.glob(INPUT_PATTERN))

for f in files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0] if 'study' in adata.obs.columns else f
    print(f"{study_name}  –  {adata.n_obs:,} cells, {adata.n_vars:,} genes")
    
    for col in sorted(adata.obs.columns):
        if pd.api.types.is_numeric_dtype(adata.obs[col]):
            continue
        
        vals = adata.obs[col].dropna().unique()
        n_unique = len(vals)
        
        if n_unique <= 25:
            val_str = ', '.join(str(v) for v in sorted(vals))
        else:
            val_str = f"{n_unique} unique values – first 15: " + ', '.join(str(v) for v in sorted(vals)[:15])
        
        print(f" {col}: {val_str}")
    
    print()

Geistlinger2020  –  38,965 cells, 15,328 genes
 cancer_type: Ovarian Cancer
 cell_cycle_phase: G1/S, G2/M, Intermediate, Not cycling, nan
 cell_subtype: B_cell, Endothelial, Epithelial, Fibroblast, Macrophage, Malignant, NK_cell, Pericyte, Plasma_cell, T_cell
 cell_type: B_cell, Endothelial, Epithelial, Fibroblast, Macrophage, Malignant, NK_cell, Pericyte, Plasma, T_cell
 ct_response: refractory, resistant, sensitive
 encode.celltype: Adipocytes, Astrocytes, B-cells, CD4+ T-cells, CD8+ T-cells, Chondrocytes, DC, Endothelial cells, Epithelial cells, Erythrocytes, Fibroblasts, HSC, Keratinocytes, Macrophages, Mesangial cells, Monocytes, Myocytes, NK cells, Pericytes, Skeletal muscle
 hpca.celltype: 30 unique values – first 15: Astrocyte, BM, B_cell, CMP, Chondrocytes, DC, Embryonic_stem_cells, Endothelial_cells, Epithelial_cells, Fibroblasts, GMP, HSC_CD34+, Hepatocytes, Keratinocytes, MEP
 mp_assignment: 51 unique values – first 15: CAF1, CAF2, CAF4, CAF6, CD4 - Cell Cycle, CD4 - Cytoto

In [20]:
INPUT_PATTERN = "*.h5ad"  
OUTPUT_FILE = "ovarian.h5ad"

valid_prefixes = [
    'Geistlinger2020', 'Izar2020_10X', 'Izar2020_SmartSeq2',
    'Nath2021_10X', 'Nath2021_iCell8', 'Olalekan2021', 'Olbrecht2021',
    'Qian2020', 'Regner2021', 'Shih2018', 'Tang-Huau2018',
    'Zhang2019', 'Zhang2022'
]
all_files = sorted(glob.glob(INPUT_PATTERN))
real_files = [f for f in all_files if any(f.startswith(p) for p in valid_prefixes)]
print(f"Found {len(real_files)} ovarian study files (ignored {len(all_files)-len(real_files)} other files)\n")


def label_malignant(adata, study_name):

    if 'cell_type' in adata.obs.columns:
        ct = adata.obs['cell_type'].astype(str).str.lower().str.strip()
        if ct.str.contains('malignant', na=False).any():
            return ct.str.contains('malignant', na=False), "cell_type='Malignant'"

    if 'disease' in adata.obs.columns:
        disease = adata.obs['disease'].astype(str).str.lower().str.strip()
        disease_kw = ['cancer', 'carcinoma', 'tumor', 'malignant', 'sarcoma', 'neoplasm']
        mask = disease.str.contains('|'.join(disease_kw), na=False)
        if mask.any():
            return mask, f"disease column: {disease[mask].unique()[:3]}"
    
    if 'source' in adata.obs.columns and 'cell_type' in adata.obs.columns:
        src = adata.obs['source'].astype(str).str.lower().str.strip()
        ct = adata.obs['cell_type'].astype(str).str.lower().str.strip()
        is_target = ct.str.contains('epithelial|tumor', na=False)
        is_tumor_source = src.str.contains('tumor|malignant|cancer', na=False)
        mask = is_target & is_tumor_source
        if mask.any():
            return mask, "source + epithelial"
    
    return pd.Series(False, index=adata.obs.index), "all non‑malignant"

temp_files = []
total_malig = 0
total_non = 0

for f in real_files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0]
    
    if 'cancer_type' not in adata.obs.columns or adata.obs['cancer_type'].isna().all():

        adata.obs['cancer_type'] = 'Ovarian Cancer'
   
        print(f" {study_name}: cancer_type missing/NaN – set to 'Ovarian Cancer'")
    
    malignant_mask, strategy = label_malignant(adata, study_name)
    adata.obs['is_malignant'] = malignant_mask.astype(int)
    
    malig_count = malignant_mask.sum()
    non_count = (~malignant_mask).sum()
    total_malig += malig_count
    total_non += non_count
    
    print(f"{study_name:<25s}  {adata.n_obs:>7,} cells  →  malignant: {malig_count:>6,}   non‑malignant: {non_count:>7,}   ({strategy})")
    
    temp_file = f"__temp_{study_name}.h5ad"
    adata.write_h5ad(temp_file)
    temp_files.append(temp_file)

print(f"TOTAL: {total_malig+total_non:>7,} cells  →  malignant: {total_malig:>6,}   non‑malignant: {total_non:>7,}")

print(f"\nMerging {len(temp_files)} studies with inner gene join...")
adatas = [sc.read_h5ad(tf) for tf in temp_files]
adata_all = adatas[0].concatenate(
    adatas[1:],
    batch_key='study',
    join='inner',
    index_unique='-'
)
print(f"Merged: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

study_names_original = []
for tf in temp_files:
    ad = sc.read_h5ad(tf)
    study_names_original.append(ad.obs['study'].iloc[0])
study_map = {str(i): name for i, name in enumerate(study_names_original)}
adata_all.obs['study'] = adata_all.obs['study'].astype(str).map(study_map)
print(f"Study names restored: {sorted(adata_all.obs['study'].unique())}")

essential = ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']
keep = [c for c in essential if c in adata_all.obs.columns]
adata_all.obs = adata_all.obs[keep]
print(f"Metadata retained: {list(adata_all.obs.columns)}")

print("\nCancer types:", sorted(adata_all.obs['cancer_type'].astype(str).unique()))
print("Malignant distribution:")
print(adata_all.obs['is_malignant'].value_counts())

# Drop NaN if any
for col in keep:
    n_nan = adata_all.obs[col].isna().sum()
    if n_nan > 0:
        print(f"{col}: {n_nan} NaN – dropping")
        adata_all = adata_all[~adata_all.obs[col].isna(), :].copy()

for col in adata_all.obs.columns:
    if adata_all.obs[col].dtype == 'object':
        adata_all.obs[col] = adata_all.obs[col].astype(str)

print(f"Final clean dataset: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")
adata_all.write_h5ad(OUTPUT_FILE)

for tf in temp_files:
    os.remove(tf)

Found 13 ovarian study files (ignored 0 other files)

Geistlinger2020             38,965 cells  →  malignant:  5,618   non‑malignant:  33,347   (cell_type='Malignant')
Izar2020_10X                 9,306 cells  →  malignant:    704   non‑malignant:   8,602   (cell_type='Malignant')
Izar2020_SmartSeq2             125 cells  →  malignant:    117   non‑malignant:       8   (cell_type='Malignant')
Nath2021_10X                23,888 cells  →  malignant: 17,206   non‑malignant:   6,682   (cell_type='Malignant')
Nath2021_iCell8                148 cells  →  malignant:    148   non‑malignant:       0   (cell_type='Malignant')
Olalekan2021                 9,717 cells  →  malignant:  4,050   non‑malignant:   5,667   (cell_type='Malignant')
Olbrecht2021                13,464 cells  →  malignant:  2,769   non‑malignant:  10,695   (cell_type='Malignant')
Qian2020                    15,520 cells  →  malignant:  6,485   non‑malignant:   9,035   (cell_type='Malignant')
Regner2021                  55,449

/tmp/ipykernel_4870/1626225034.py:72: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adatas[0].concatenate(


Merged: 231,791 cells × 9,511 genes
Study names restored: ['Geistlinger2020', 'Izar2020_10X', 'Izar2020_SmartSeq2', 'Nath2021_10X', 'Nath2021_iCell8', 'Olalekan2021', 'Olbrecht2021', 'Qian2020', 'Regner2021', 'Shih2018', 'Tang-Huau2018', 'Zhang2019', 'Zhang2022']
Metadata retained: ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']

Cancer types: ['Carcinosarcoma', 'Endometrial Cancer', 'Gastrointestinal Stromal Tumor', 'Normal', 'Ovarian Cancer', 'Peritoneal Cancer', 'nan']
Malignant distribution:
is_malignant
0    163992
1     67799
Name: count, dtype: int64
Final clean dataset: 231,791 cells × 9,511 genes
